# Notebook 2 — Feature Engineering & Rule-Based Labeling (Layer 1 Interlock)
### Immersion Aluminium Holding Furnace (1120 kg/ch) — Anomaly Detection Pipeline

**Purpose:** Load `cleaned_data.csv`, engineer physics-informed features, and apply
the **Layer 1 rule-based interlock logic** to generate the `is_anomaly` target column
that Notebook 3 will use for supervised training and ensemble validation.


In [ ]:
# =========================================================
# 0. ENVIRONMENT SETUP
# =========================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
pd.set_option("display.max_columns", None)

INPUT_PATH = "/kaggle/input/notebooks/aliciakyoumi/m15-cia-01-cleansing"
OUTPUT_PATH = "/kaggle/working/"
CORE_FEATURES = ["molten_temp", "heater1", "heater2", "voltage_avr", "current_avr", "power_total"]

df = pd.read_csv(os.path.join(INPUT_PATH, "cleaned_data.csv"), parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print("Loaded cleaned data:", df.shape)
df.head()


## 1. Feature Engineering

In [ ]:
# =========================================================
# 1a. Resistance (R = P / I^2)
# Physical relationship: Power = I^2 * R  ->  R = Power / I^2
# Divide-by-zero (current_avr == 0) is set to NaN/Inf, never fabricated.
# =========================================================
with np.errstate(divide="ignore", invalid="ignore"):
    df["resistance"] = np.where(
        df["current_avr"] == 0,
        np.nan,
        df["power_total"] / (df["current_avr"] ** 2)
    )

print("Resistance summary:")
print(df["resistance"].describe())
print("\nNaN resistance rows (current_avr == 0):", df["resistance"].isna().sum())


In [ ]:
# =========================================================
# 1b. Bin resistance into categories: Very High / Normal / Very Low
# Thresholds derived from the 5th / 95th percentile of valid (non-null) resistance values.
# Adjust these thresholds if plant engineers provide calibrated setpoints.
# =========================================================
valid_r = df["resistance"].dropna()
low_thresh = valid_r.quantile(0.05)
high_thresh = valid_r.quantile(0.95)
print(f"Resistance bin thresholds -> Low: {low_thresh:.4f}, High: {high_thresh:.4f}")

def bin_resistance(r):
    if pd.isna(r):
        return "Unknown"
    elif r < low_thresh:
        return "Very Low"
    elif r > high_thresh:
        return "Very High"
    else:
        return "Normal"

df["resistance_class"] = df["resistance"].apply(bin_resistance)
df["resistance_class"].value_counts()


In [ ]:
# =========================================================
# 1b2. Deteksi Sensor Fault pada molten_temp (nilai 0)
# =========================================================
# Aluminium cair pada holding furnace beroperasi di rentang 640-660°C (lihat
# C.3 Ringkasan Domain Knowledge). Pembacaan molten_temp = 0°C SECARA FISIS
# TIDAK MUNGKIN terjadi selama furnace beroperasi normal -> ini indikasi
# SENSOR/THERMOCOUPLE TIDAK TERBACA (fault), BUKAN penurunan suhu riil.
#
# Jika nilai 0 ini tidak ditangani, ia akan menghasilkan delta_temp_1h palsu
# yang sangat ekstrem (mis. 650°C -> 0°C = -650°C), yang bisa salah
# terdeteksi sebagai Thermal Lag atau Low Temp - Eutectic Risk, padahal
# sebenarnya cuma sensor error.
# =========================================================
df["molten_temp_sensor_fault_flag"] = (df["molten_temp"] == 0).astype(int)

# molten_temp_valid: versi "bersih" dari molten_temp -> NaN saat sensor fault,
# dipakai untuk SEMUA perhitungan delta/threshold turunan (kolom molten_temp
# asli TIDAK diubah, data mentah tetap dipertahankan apa adanya).
df["molten_temp_valid"] = df["molten_temp"].where(df["molten_temp"] > 0)

print("Baris dengan molten_temp = 0 (indikasi sensor fault):",
      df["molten_temp_sensor_fault_flag"].sum(),
      f"({df['molten_temp_sensor_fault_flag'].mean()*100:.4f}% dari data)")


In [ ]:
# =========================================================
# 1c. delta_temp_1h — rolling 1-hour temperature change
# Assumes a roughly regular sampling interval; infer it from the timestamp deltas.
# Menggunakan molten_temp_valid (bukan molten_temp mentah) agar baris dengan
# sensor fault (molten_temp = 0, lihat 1b2) tidak menghasilkan delta palsu
# yang ekstrem.
# =========================================================
median_interval = df["timestamp"].diff().median()
print("Median sampling interval:", median_interval)

samples_per_hour = max(1, int(pd.Timedelta("1h") / median_interval))
print("Approx samples per hour (jumlah interval dalam 1 jam):", samples_per_hour)

df["delta_temp_1h"] = df["molten_temp_valid"] - df["molten_temp_valid"].shift(samples_per_hour)
# NaN muncul jika titik AWAL atau AKHIR window kena sensor fault -> tidak bisa
# dievaluasi secara valid, diisi 0 (netral: "tidak ada perubahan yang bisa
# dikonfirmasi"), BUKAN diasumsikan sebagai penurunan/kenaikan ekstrem.
df["delta_temp_1h"] = df["delta_temp_1h"].fillna(0)

df["delta_temp_1h"].describe()


In [ ]:
# =========================================================
# 1d. Heater status flags
# =========================================================
df["heater_active"]    = ((df["heater1"] > 0) | (df["heater2"] > 0)).astype(int)
df["heater_both_off"]  = ((df["heater1"] == 0) & (df["heater2"] == 0)).astype(int)

print(df[["heater_active", "heater_both_off"]].mean().rename("proportion_1"))


In [ ]:
# =========================================================
# 1e. Additional helper features for rule logic
# =========================================================
samples_per_hour = max(1, int(pd.Timedelta("1h") / median_interval))

df["current_avr_roll_std"]  = df["current_avr"].rolling(window=samples_per_hour, min_periods=1).std().fillna(0)

# Pelonggaran current_spike: Ubah pengali dari 3 * std menjadi 2 * std agar lebih sensitif terhadap lonjakan arus
df["current_avr_spike"]     = (df["current_avr"] > (df["current_avr"].rolling(window=samples_per_hour*3, min_periods=1).mean()
                                                       + 2 * df["current_avr"].rolling(window=samples_per_hour*3, min_periods=1).std())).fillna(False)

df["power_total_roll_std"]  = df["power_total"].rolling(window=samples_per_hour, min_periods=1).std().fillna(0)

# Pelonggaran power_stagnant: Naikkan kuartil dari 10% ke 20% agar kondisi daya datar lebih mudah tertangkap
df["power_stagnant"]        = df["power_total_roll_std"] < df["power_total_roll_std"].quantile(0.20)

print("current_avr_spike rate:", df['current_avr_spike'].mean())
print("power_stagnant rate:  ", df['power_stagnant'].mean())

In [ ]:
# =========================================================
# 1f. Fitur Titik Eutektik & Liquidus — Diagram Fasa Aluminium-Silikon
# =========================================================
# CATATAN TEORI (koreksi terhadap versi sebelumnya):
# Diagram fasa Al-Si punya DUA titik acuan yang berbeda secara fisis:
#
#   1) EUTECTIC_TEMP_C = 577 C -> titik invarian eutektik, suhu TETAP
#      (tidak bergantung %Si). Di bawah suhu ini, SISA logam cair yang masih
#      ada akan membeku total (reaksi eutektik L -> alpha-Al + Si). Ini
#      adalah "titik tanpa jalan kembali" -> sangat berbahaya bagi pompa,
#      elemen pemanas, dan nozzle bila benar-benar tercapai.
#
#   2) LIQUIDUS_TEMP_C -> suhu MULAI terbentuknya kristal padat primer
#      alpha-Al (awal fase mush/semi-solid). Nilai ini BERGANTUNG pada %Si
#      paduan (mengikuti kurva liquidus, bukan titik tetap). Karena sensor
#      komposisi (%Si) tidak tersedia di dataset ini, dipakai NILAI ASUMSI
#      (default 615 C) yang mewakili paduan casting Al-Si umum (~7-9% Si,
#      spt A356/A380) yang lazim dipakai pada immersion holding furnace.
#      -> SESUAIKAN nilai ini bila komposisi paduan aktual diketahui
#      (mis. dari Certificate of Analysis/COA supplier ingot).
#
# Versi SEBELUMNYA hanya memakai satu angka (645 C = 577 + margin 68 C)
# sbg "eutectic risk". Itu sebetulnya lebih dekat ke ambang DEVIASI SETPOINT
# (650 C - 5 C) daripada representasi fisis diagram fasa yang sesungguhnya,
# karena 645 C masih ~30-40 C DI ATAS liquidus (logam masih 100% cair).
# Di bawah ini ditambahkan fitur yang selaras dgn teori diagram fasa yang
# benar: distance_to_liquidus (awal mush), distance_to_eutectic (beku total),
# dan estimasi fraksi padat primer via lever rule.
# =========================================================
# CATATAN: menggunakan molten_temp_valid (bukan molten_temp mentah) agar
# baris dengan sensor fault (molten_temp = 0, lihat 1b2) tidak salah memicu
# "Low Temp - Eutectic Risk" / "Mush Formation" secara palsu.
EUTECTIC_TEMP_C        = 577   # Titik eutektik Al-Si (invarian, TETAP terhadap komposisi)
LIQUIDUS_TEMP_C        = 615   # ASUMSI liquidus paduan casting umum (~7-9% Si) -> sesuaikan bila komposisi aktual diketahui
EUTECTIC_SAFETY_MARGIN = 15    # Buffer alarm DI ATAS titik eutektik murni (agar alarm CRITICAL tidak baru menyala tepat saat sudah beku)

# --- Fitur jarak (selaras definisi fisis diagram fasa) ---
df["distance_to_liquidus"] = df["molten_temp_valid"] - LIQUIDUS_TEMP_C   # < 0 -> sudah masuk mushy zone (mulai ada kristal padat)
df["distance_to_eutectic"] = df["molten_temp_valid"] - EUTECTIC_TEMP_C   # <= 0 -> mendekati/sudah beku total (kolom lama, dipertahankan utk downstream)

# --- Flag zona mushy (semi-solid): liquidus > temp >= eutectic + margin ---
# Kristal primer alpha-Al MULAI terbentuk (viskositas naik, risiko sludge/
# dross menumpuk, aliran logam mulai tidak lancar) TAPI logam belum beku total.
df["mushy_zone_flag"] = (
    (df["molten_temp_valid"] < LIQUIDUS_TEMP_C) &
    (df["molten_temp_valid"] >= EUTECTIC_TEMP_C + EUTECTIC_SAFETY_MARGIN)
).astype(int)

# --- Flag mendekati/melewati titik eutektik murni (risiko beku total) ---
df["eutectic_risk_flag"] = (df["molten_temp_valid"] < EUTECTIC_TEMP_C + EUTECTIC_SAFETY_MARGIN).astype(int)

# --- Estimasi fraksi padat primer (lever rule / pendekatan linear equilibrium) ---
# Valid untuk molten_temp di antara eutectic dan liquidus (equilibrium
# approximation, mengabaikan efek non-equilibrium/segregasi ala Scheil).
# Di luar rentang itu di-clip: 0 = masih 100% cair, 1 = sudah beku total.
df["est_primary_solid_fraction"] = (
    (LIQUIDUS_TEMP_C - df["molten_temp_valid"]) / (LIQUIDUS_TEMP_C - EUTECTIC_TEMP_C)
).clip(lower=0, upper=1)

print("distance_to_liquidus summary:")
print(df["distance_to_liquidus"].describe())
print("\ndistance_to_eutectic summary:")
print(df["distance_to_eutectic"].describe())
print("\nBaris di zona mushy (semi-solid, kristal primer alpha-Al mulai terbentuk):",
      df["mushy_zone_flag"].sum())
print("Baris mendekati/melewati titik eutektik murni (risiko beku total, <",
      EUTECTIC_TEMP_C + EUTECTIC_SAFETY_MARGIN, "C):", df["eutectic_risk_flag"].sum())


## 2. Rule-Based Labeling (Layer 1 Interlock Logic)

Each physical failure mode below is encoded as an explicit boolean rule.
`is_anomaly = 1` if **any** rule fires.


In [ ]:
# =========================================================
# 2. RULE-BASED LABELING & ROOT CAUSE (Layer 1 Interlock Logic)
# Konsolidasi: SATU kolom "reason" (akar masalah) + SATU kolom "severity_level"
# Tidak ada lagi kolom boolean rule_* terpisah yang disimpan ke df — setiap
# sub-kondisi hanya dihitung sebagai variabel sementara (in-memory), lalu
# digabung lewat np.select() menjadi target akhir.
# =========================================================

# --- a) Uncontrolled Heating: suhu naik padahal KEDUA heater mati ---
cond_uncontrolled_heating = (df["delta_temp_1h"] > 0) & (df["heater_both_off"] == 1)

# --- b) Short Circuit: resistansi sangat rendah + power stagnan + current spike ---
valid_r = df["resistance"].dropna()
low_thresh_relaxed = valid_r.quantile(0.10)
cond_short_circuit = (
    (df["resistance"] < low_thresh_relaxed) &
    (df["power_stagnant"]) &
    (df["current_avr_spike"])
)

# --- c) Thermal Lag: suhu turun DAN TERUS TURUN sepanjang window 1 jam
# setelah heater ON (bukan cuma dicek satu langkah setelahnya) ---
# CATATAN (constraint eksplisit dipertahankan): threshold delta_temp_1h <= -5
# TIDAK diubah menjadi -10 meskipun Ringkasan Domain Knowledge menyebut ambang
# ahli -10°C. -5 adalah keputusan eksplisit tim sebelumnya (lebih sensitif) dan
# sengaja TIDAK di-auto-correct di sini.
#
# Definisi ahli (Skenario B.#4): "Dalam window 1 jam: molten_temp turun,
# heater berubah status ke ON, namun SETELAH ON molten_temp TETAP TURUN TERUS
# (tidak ada respons kenaikan)." Artinya evaluasi TIDAK CUKUP hanya
# membandingkan satu langkah berikutnya -> harus dicek di seluruh window 1 jam
# (samples_per_hour = 4 x interval 15 menit), memastikan TIDAK ADA satu pun
# kenaikan (rebound) suhu di dalamnya. molten_temp_valid dipakai (bukan
# molten_temp mentah) agar baris sensor fault (1b2) tidak menghasilkan
# false-rebound/false-drop.
heater_turned_on = (df["heater_active"] == 1) & (df["heater_active"].shift(1) == 0)

temp_diff_step = df["molten_temp_valid"].diff()  # perubahan per-langkah (t vs t-1)

# future_max_step_diff[i] = kenaikan TERBESAR pada langkah manapun di dalam
# (samples_per_hour - 1) langkah SETELAH baris i. Jika nilainya <= 0, berarti
# TIDAK ADA kenaikan sama sekali sepanjang window tsb -> suhu "tetap turun
# terus" persis seperti definisi ahli.
future_max_step_diff = (
    temp_diff_step.shift(-1)
    .iloc[::-1]
    .rolling(window=samples_per_hour - 1, min_periods=1)
    .max()
    .iloc[::-1]
)
no_recovery_within_1h = future_max_step_diff <= 0

cond_thermal_lag = (
    (df["delta_temp_1h"] <= -5) &                       # <-- threshold DIPERTAHANKAN
    heater_turned_on.fillna(False) &
    no_recovery_within_1h.fillna(False)                  # suhu tidak pernah rebound sepanjang window 1 jam
)

print("Thermal Lag terdeteksi (versi window 1 jam penuh):", cond_thermal_lag.sum(),
      "baris  |  (versi 1-langkah lama akan menghasilkan jumlah lebih tinggi/longgar)")

# --- d) Overheat ---
cond_overheat = df["molten_temp_valid"] > 670

# --- d2) Metal Temperature Low (Skenario Sela) ---
# Suhu sudah di bawah rentang operasional normal (< 640 C) namun logam masih
# 100% cair (>= LIQUIDUS_TEMP_C) -> belum masuk mushy zone / eutectic risk,
# jadi ini "sela" peringatan dini sebelum kondisi tsb tereskalasi.
cond_metal_temp_low = (
    (df["molten_temp_valid"] < 640) &
    (df["molten_temp_valid"] >= LIQUIDUS_TEMP_C)
)

# --- e) Low Temp - Eutectic Risk (mendekati/melewati titik BEKU TOTAL Al-Si) ---
# Menggunakan eutectic_risk_flag dari cell 1f: molten_temp_valid < 592°C
cond_low_temp_eutectic = df["eutectic_risk_flag"] == 1

# --- f) Starvation Power / Voltage Drop (Anomali Multivariat, Skenario B.#11) ---
# Thermal Lag TERJADI BERSAMAAN dengan voltage_avr turun/naik di luar toleransi
# ±10% dari nominal 380V -> power_total turun/stagnan meski heater berstatus ON.
VOLTAGE_NOMINAL    = 380
VOLTAGE_LOW_LIMIT  = VOLTAGE_NOMINAL * 0.9   # 342V
VOLTAGE_HIGH_LIMIT = VOLTAGE_NOMINAL * 1.1   # 418V

voltage_out_of_tolerance = (
    (df["voltage_avr"] < VOLTAGE_LOW_LIMIT) |
    (df["voltage_avr"] > VOLTAGE_HIGH_LIMIT)
)
cond_voltage_drop_lag = cond_thermal_lag & voltage_out_of_tolerance

# --- g) Approaching Liquidus - Mush Formation (sesuai teori diagram fasa Al-Si) ---
# Kristal primer alpha-Al mulai terbentuk (592°C <= molten_temp_valid < 615°C):
# logam belum beku total, tapi viskositas mulai naik & berisiko sludge/dross
# menumpuk. Sinyal PERINGATAN DINI sebelum benar-benar mencapai titik eutektik.
cond_mushy_zone = df["mushy_zone_flag"] == 1

# --- h) Sensor Fault: molten_temp = 0 (lihat cell 1b2) ---
# Bukan anomali PROSES, tapi anomali DATA QUALITY -> tetap perlu di-flag agar
# tim maintenance tahu ada sensor/thermocouple yang perlu dicek/kalibrasi,
# dan supaya baris ini TIDAK diam-diam dianggap "Normal Operation" (yang
# menyesatkan karena sebenarnya kondisi furnace pada baris ini tidak terbaca).
cond_molten_temp_sensor_fault = df["molten_temp_sensor_fault_flag"] == 1

# =========================================================
# Konsolidasi -> kolom "reason" & "severity_level"
# Urutan list "conditions" di bawah ini ADALAH urutan prioritas: kondisi yang
# lebih atas selalu menang bila beberapa kondisi terpicu bersamaan pada baris
# yang sama (np.select mengambil match PERTAMA yang True). Sinyal-sinyal
# kelistrikan (short circuit, uncontrolled heating, voltage drop) TIDAK
# bergantung pada molten_temp sehingga tetap valid diperiksa lebih dulu
# meskipun molten_temp sedang sensor-fault pada baris yang sama.
#
#   1) Short Circuit                        -> FATAL/EMERGENCY (hard interlock mutlak)
#   2) Starvation Power / Voltage Drop       -> CRITICAL (Skenario B.#11)
#   3) Uncontrolled Heating                  -> CRITICAL (Skenario B.#1)
#   4) Overheat                              -> CRITICAL (Skenario B.#5)
#   5) Metal Temperature Low (Skenario Sela) -> CRITICAL (< 640°C, masih cair >= liquidus)
#   6) Low Temp - Eutectic Risk              -> CRITICAL (mendekati beku total, <592°C)
#   7) Approaching Liquidus - Mush Formation -> WARNING  (awal solidifikasi, 592-615°C)
#   8) Thermal Lag (window 1 jam penuh)      -> WARNING  (Skenario B.#4)
#   9) Sensor Fault - Molten Temp Unreadable -> WARNING  (data quality, bukan anomali proses)
#   10) Selain semua di atas                 -> NORMAL
# =========================================================
conditions = [
    cond_short_circuit,
    cond_voltage_drop_lag,
    cond_uncontrolled_heating,
    cond_overheat,
    cond_metal_temp_low,
    cond_low_temp_eutectic,
    cond_mushy_zone,
    cond_thermal_lag,
    cond_molten_temp_sensor_fault,
]

reason_choices = [
    "Short Circuit",
    "Starvation Power / Voltage Drop",
    "Uncontrolled Heating",
    "Overheat",
    "Metal Temperature Low",
    "Low Temp - Eutectic Risk",
    "Approaching Liquidus - Mush Formation",
    "Thermal Lag",
    "Sensor Fault - Molten Temp Unreadable",
]

severity_choices = [
    "FATAL/EMERGENCY",
    "CRITICAL",
    "CRITICAL",
    "CRITICAL",
    "CRITICAL",
    "CRITICAL",
    "WARNING",
    "WARNING",
    "WARNING",
]

df["reason"]         = np.select(conditions, reason_choices, default="Normal Operation")
df["severity_level"] = np.select(conditions, severity_choices, default="NORMAL")

# is_anomaly tetap dipertahankan sebagai target biner utama untuk Model #1-#3
# (lihat Bagian D Ringkasan Domain Knowledge) -> diturunkan langsung dari severity_level
df["is_anomaly"] = (df["severity_level"] != "NORMAL").astype(int)

print("Total Anomali:", df["is_anomaly"].sum(),
      f"({df['is_anomaly'].mean()*100:.4f}% dari {len(df)} baris)")

print("\nDistribusi reason (root cause utama - single label, prioritas tertinggi):")
display(df["reason"].value_counts().to_frame("count"))

print("\nDistribusi severity_level:")
display(df["severity_level"].value_counts().to_frame("count"))

# =========================================================
# Kolom diagnostik tambahan: all_triggered_reasons
# "reason" di atas HANYA mengambil match PERTAMA berdasarkan prioritas
# severity (np.select) -> untuk SATU baris, hanya SATU reason yang tercatat
# sbg reason utama, meskipun bisa jadi ADA kondisi lain yang secara teknis
# juga True di baris yang sama (mis. overheat bersamaan dgn thermal lag).
# Kolom ini mencatat SEMUA kondisi yang benar-benar terpicu di baris tsb,
# supaya engineer bisa cek root cause majemuk tanpa mengubah "reason"
# (single-label) yang dipakai sbg target model.
# =========================================================
all_conditions_named = {
    "Short Circuit": cond_short_circuit,
    "Starvation Power / Voltage Drop": cond_voltage_drop_lag,
    "Uncontrolled Heating": cond_uncontrolled_heating,
    "Overheat": cond_overheat,
    "Metal Temperature Low": cond_metal_temp_low,
    "Low Temp - Eutectic Risk": cond_low_temp_eutectic,
    "Approaching Liquidus - Mush Formation": cond_mushy_zone,
    "Thermal Lag": cond_thermal_lag,
    "Sensor Fault - Molten Temp Unreadable": cond_molten_temp_sensor_fault,
}

flags_df = pd.DataFrame(all_conditions_named)
df["all_triggered_reasons"] = flags_df.apply(
    lambda row: ", ".join(row.index[row]) if row.any() else "Normal Operation", axis=1
)
df["n_triggered_reasons"] = flags_df.sum(axis=1)

print("\nBaris dengan LEBIH DARI SATU root cause terpicu bersamaan:",
      (df["n_triggered_reasons"] > 1).sum())
if (df["n_triggered_reasons"] > 1).sum() > 0:
    print("Contoh kombinasi root cause majemuk (5 teratas):")
    display(
        df.loc[df["n_triggered_reasons"] > 1, "all_triggered_reasons"]
        .value_counts().head(5).to_frame("count")
    )


In [ ]:
# =========================================================
# 2b. Encoding severity_level untuk Target Model Multi-Class (Notebook 3)
# severity_level (4 kelas: NORMAL/WARNING/CRITICAL/FATAL-EMERGENCY) akan
# dipakai sebagai target multi-class Model #3 (Gradient Boosted Trees),
# selain is_anomaly biner yang tetap dipertahankan untuk Model #1 (OCSVM)
# & Model #2 (LOF) yang unsupervised. Lihat Bagian D Ringkasan Domain
# Knowledge (D.1, D.3, D.4).
# =========================================================
SEVERITY_ORDER_RANK = ["NORMAL", "WARNING", "CRITICAL", "FATAL/EMERGENCY"]
severity_rank_map = {level: rank for rank, level in enumerate(SEVERITY_ORDER_RANK)}

# Encoding ORDINAL (0=NORMAL ... 3=FATAL/EMERGENCY) -> berguna untuk model yang
# ingin memanfaatkan urutan keparahan (mis. ordinal classifier / cost-sensitive
# loss), dan untuk stratifikasi cross-validation di Notebook 3.
df["severity_rank"] = df["severity_level"].map(severity_rank_map).astype(int)

# Cek distribusi & rasio imbalance antar 4 kelas (penting untuk D.3 -> SMOTE /
# scale_pos_weight / class_weight di Notebook 3, karena FATAL/EMERGENCY &
# CRITICAL cenderung rare event dibanding NORMAL).
severity_dist = df["severity_level"].value_counts().reindex(SEVERITY_ORDER_RANK).fillna(0).astype(int)
severity_pct  = (severity_dist / len(df) * 100).round(4)

summary = pd.DataFrame({
    "severity_level": SEVERITY_ORDER_RANK,
    "severity_rank": [severity_rank_map[l] for l in SEVERITY_ORDER_RANK],
    "count": severity_dist.values,
    "pct_of_total": severity_pct.values,
})
display(summary)

majority_count = severity_dist.max()
print("\nRasio imbalance tiap kelas terhadap kelas mayoritas (NORMAL):")
for level in SEVERITY_ORDER_RANK:
    cnt = severity_dist[level]
    ratio = majority_count / cnt if cnt > 0 else float("inf")
    print(f"  {level:<18} n={cnt:<8} -> 1 : {ratio:,.1f}")


### Catatan untuk Notebook 3 (Modeling)

`severity_level` (4 kelas) & `severity_rank` (encoding ordinal 0–3) di atas disiapkan sebagai **target multi-class** untuk Model #3 (Gradient Boosted Trees / ordinal classifier), sedangkan `is_anomaly` (biner) tetap dipakai untuk Model #1 (One-Class SVM) & Model #2 (LOF) yang unsupervised — sesuai Bagian D Ringkasan Domain Knowledge.

Karena kelas `CRITICAL` dan terutama `FATAL/EMERGENCY` kemungkinan besar *rare event* (lihat rasio imbalance di atas), pertimbangkan di Notebook 3:
- **Stratified split / stratified k-fold** berdasarkan `severity_level`, bukan `is_anomaly`, agar kelas minoritas tetap terwakili di setiap fold.
- **class_weight="balanced"** atau **scale_pos_weight** setara multi-class (mis. `sample_weight` custom), sebagai alternatif yang lebih aman daripada oversampling sintetis untuk data time-series sensor.
- Jika tetap memakai **SMOTE/oversampling**, lakukan **setelah** train-test split (bukan sebelum) agar tidak terjadi data leakage antara train dan test set.
- Evaluasi dengan **macro/weighted F1** dan **confusion matrix per kelas**, bukan hanya accuracy — karena distribusi kelas sangat timpang.


## 3. Visualization

In [ ]:
# =========================================================
# 3a. Time-series dengan anomali diwarnai berdasarkan Severity Level
# FATAL/EMERGENCY = hitam (marker besar), CRITICAL = merah, WARNING = oranye
# Garis referensi liquidus & eutektik ditambahkan untuk konteks diagram fasa.
# =========================================================
SEVERITY_STYLE = {
    "FATAL/EMERGENCY": {"color": "black",  "size": 11},
    "CRITICAL":         {"color": "red",    "size": 7},
    "WARNING":           {"color": "orange", "size": 5},
}
SEVERITY_ORDER = ["WARNING", "CRITICAL", "FATAL/EMERGENCY"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["timestamp"], y=df["molten_temp"], mode="lines",
                          name="molten_temp", line=dict(color="steelblue", width=1)))

for level in SEVERITY_ORDER:
    sub = df[df["severity_level"] == level]
    style = SEVERITY_STYLE[level]
    fig.add_trace(go.Scatter(
        x=sub["timestamp"], y=sub["molten_temp"], mode="markers",
        name=level,
        marker=dict(color=style["color"], size=style["size"], symbol="circle"),
        text=sub["reason"], hovertemplate="%{x}<br>molten_temp=%{y}<br>reason=%{text}<extra></extra>"
    ))

# Garis referensi diagram fasa Al-Si
fig.add_hline(y=LIQUIDUS_TEMP_C, line_dash="dash", line_color="gray",
              annotation_text=f"Liquidus (asumsi) {LIQUIDUS_TEMP_C}°C", annotation_position="top left")
fig.add_hline(y=EUTECTIC_TEMP_C, line_dash="dot", line_color="darkred",
              annotation_text=f"Eutektik (invarian) {EUTECTIC_TEMP_C}°C", annotation_position="bottom left")

# Garis batas rentang operasional mesin (Overheat / Low Temp Limit)
fig.add_hline(y=670, line_dash="dash", line_color="darkorange",
              annotation_text="Overheat Limit 670°C", annotation_position="top right")
fig.add_hline(y=640, line_dash="dash", line_color="darkorange",
              annotation_text="Low Temp Limit 640°C", annotation_position="bottom right")

fig.update_layout(title="molten_temp Timeline — Anomali berdasarkan Severity Level (Layer 1 Rules)",
                   height=500, xaxis_title="timestamp", yaxis_title="molten_temp")
fig.show()


In [ ]:
# =========================================================
# 3b. Normal vs Anomaly class distribution
# =========================================================
class_counts = df["is_anomaly"].value_counts().rename({0: "Normal", 1: "Anomaly"}).reset_index()
class_counts.columns = ["class", "count"]

fig = px.bar(class_counts, x="class", y="count", color="class", text="count",
             title="Class Distribution — Normal vs Anomaly (Layer 1 Labels)",
             color_discrete_map={"Normal": "mediumseagreen", "Anomaly": "firebrick"})
fig.show()


In [ ]:
# =========================================================
# 3c. Distribusi Severity Level (Bar Chart & Pie Chart)
# =========================================================
severity_order = ["NORMAL", "WARNING", "CRITICAL", "FATAL/EMERGENCY"]
severity_color_map = {
    "NORMAL": "mediumseagreen",
    "WARNING": "orange",
    "CRITICAL": "red",
    "FATAL/EMERGENCY": "black",
}

severity_counts = (
    df["severity_level"]
    .value_counts()
    .reindex(severity_order)
    .fillna(0)
    .astype(int)
    .reset_index()
)
severity_counts.columns = ["severity_level", "count"]

# Bar chart
fig_bar = px.bar(
    severity_counts, x="severity_level", y="count", text="count",
    color="severity_level", color_discrete_map=severity_color_map,
    category_orders={"severity_level": severity_order},
    title="Distribusi Severity Level (Layer 1 Rule-Based)"
)
fig_bar.update_traces(textposition="outside")
fig_bar.show()

# Pie chart
fig_pie = px.pie(
    severity_counts, names="severity_level", values="count",
    color="severity_level", color_discrete_map=severity_color_map,
    category_orders={"severity_level": severity_order},
    title="Proporsi Severity Level (Layer 1 Rule-Based)"
)
fig_pie.show()


In [ ]:
# =========================================================
# 3e. Breakdown Root Cause (Reason) untuk baris Anomali
# Menggantikan chart "rule trigger breakdown" lama, karena kolom boolean
# rule_* terpisah sudah dikonsolidasikan menjadi kolom "reason".
# =========================================================
reason_counts = (
    df.loc[df["reason"] != "Normal Operation", "reason"]
    .value_counts()
    .reset_index()
)
reason_counts.columns = ["reason", "count"]

fig = px.bar(
    reason_counts, x="reason", y="count", text="count", color="reason",
    title="Breakdown Root Cause (Reason) — Baris Anomali"
)
fig.update_traces(textposition="outside")
fig.show()


In [ ]:
# =========================================================
# 3d. Resistance class distribution + delta_temp_1h distribution
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.countplot(
    x="resistance_class", 
    data=df, 
    order=["Very Low", "Normal", "Very High", "Unknown"],
    ax=axes[0], 
    palette="viridis",
    hue="resistance_class", 
    legend=False            
)
axes[0].set_title("Resistance Class Distribution")

sns.histplot(df["delta_temp_1h"], bins=50, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("delta_temp_1h Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 3e. Correlation heatmap of engineered features vs is_anomaly
# =========================================================
feature_cols_for_corr = CORE_FEATURES + ["resistance", "is_anomaly"]
plt.figure(figsize=(9, 7))
sns.heatmap(df[feature_cols_for_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Engineered Features Correlation")
plt.tight_layout()
plt.show()


## 4. Save Labeled Dataset

In [ ]:
# =========================================================
# 4. EXPORT
# =========================================================
out_file = os.path.join(OUTPUT_PATH, "labeled_data.csv")
df.to_csv(out_file, index=False)
print(f"Saved labeled dataset -> {out_file}")
print("Final shape:", df.shape)
print("Columns:", list(df.columns))
df.head()
